In [1]:
#imports
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [2]:
df = pd.read_csv("train.En.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df["tweet"] = df["tweet"].astype(str).fillna("")
texts = df["tweet"].str.lower()
labels = df["sarcastic"]

In [3]:
X_train_text, X_val_text, y_train, y_val = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [4]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
    sublinear_tf=True
)

In [5]:
X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)

print("train shape:", X_train.shape)
print("val shape:", X_val.shape)

train shape: (2774, 7565)
val shape: (694, 7565)


In [6]:
test_df = pd.read_csv("task_A_En_test.csv")
test_df["text"] = test_df["text"].astype(str).fillna("").str.lower()
X_test = vectorizer.transform(test_df["text"])

print("test shape:", X_test.shape)

test shape: (1400, 7565)


In [7]:
svm = LinearSVC(C=1.0, max_iter=1000, random_state=42, class_weight='balanced')
svm.fit(X_train, y_train)

LinearSVC(class_weight='balanced', random_state=42)

In [8]:
#evaluate on validation set

y_val_pred = svm.predict(X_val)

accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

print("validation results:")
print(f"accuracy: {accuracy:.4f}")
print(f"precision: {precision:.4f}")
print(f"recall: {recall:.4f}")
print(f"f1 score: {f1:.4f}")
print("\ndetailed report:")
print(classification_report(y_val, y_val_pred))

validation results:
accuracy: 0.6686
precision: 0.3293
recall: 0.3179
f1 score: 0.3235

detailed report:
              precision    recall  f1-score   support

           0       0.78      0.79      0.78       521
           1       0.33      0.32      0.32       173

    accuracy                           0.67       694
   macro avg       0.55      0.55      0.55       694
weighted avg       0.66      0.67      0.67       694



In [9]:
#final test set evaluation

y_test_pred = svm.predict(X_test)

test_accuracy = accuracy_score(test_df["sarcastic"], y_test_pred)
test_precision = precision_score(test_df["sarcastic"], y_test_pred)
test_recall = recall_score(test_df["sarcastic"], y_test_pred)
test_f1 = f1_score(test_df["sarcastic"], y_test_pred)

print("test results:")
print(f"accuracy: {test_accuracy:.4f}")
print(f"precision: {test_precision:.4f}")
print(f"recall: {test_recall:.4f}")
print(f"f1 score: {test_f1:.4f}")

test results:
accuracy: 0.7064
precision: 0.2172
recall: 0.4050
f1 score: 0.2827
